# train

## import

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import os
import sys
sys.path.append(f'{os.getcwd()}/irc_gym')
from auditoryforage.AF_env import AuditoryForaging as AF
from auditoryforage.AF_env import AuditoryForagingReward as AFR
from stable_baselines3 import PPO
from multiprocessing import Pool
import multiprocessing as mp



## train

In [6]:
food_reward_list = [1000, 1100, 1200, 1300, 1400, 1500]
att_coeff = 0.087213
att_temp = 0.25
penalty_cost = -30
time_in_game_reward = 0.
# num_epochs = 11
# seed_value = 1

## circulum

In [ ]:

# model_list={}
# for food_reward in food_reward_list:
#     # init
#     task=AF()
#     # assign values
#     task.food_reward=food_reward
#     task.time_in_game_reward=time_in_game_reward
#     task.att_coeff=att_coeff
#     task.att_temp=att_temp
#     task.penalty_cost=penalty_cost
#     # train
#     model = PPO('MlpPolicy', task, verbose=1)
#     model.learn(total_timesteps=999)
#     # save
#     model_list[food_reward]=model
#     model.save(f'./ycstore/{food_reward}')


## load and eval

In [ ]:


def performance_reward_rate(model, task, total_trials=100 ):
    '''model reward rate'''
    history_r=[]
    for _ in range(total_trials):
        t=0
        total_r=0
        x=task.reset()
        done=False
        while not done: 
            # model.predict(x)
            a,_=model.predict(x)
            x, r, done, _ = task.step(a)
            t+=1
            total_r+=r
        history_r.append((total_r/t))
    return  history_r,
        

In [ ]:
from collections import defaultdict
performances=defaultdict(list)
model_list={}
for food_reward in food_reward_list:
    # init
    task=AF()
    # assign values
    task.food_reward=food_reward
    task.time_in_game_reward=time_in_game_reward
    task.att_coeff=att_coeff
    task.att_temp=att_temp
    task.penalty_cost=penalty_cost
    # load
    model = PPO.load(f'ycstore/{food_reward}', env=task, device='cpu')
    model_list[food_reward]=model
    # eval
    # performances['rewardrate'].append(performance_reward_rate(model, task,total_trials=10000)[0])


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


# other

In [7]:
# import numpy as np
# from matplotlib import pyplot as plt
# means=np.mean(np.array(performances['rewardrate']), axis=1)
# stds=np.std(np.array(performances['rewardrate']), axis=1)*0.1
# plt.fill_between(list(range(len(means))), means - stds, means + stds, color='skyblue', alpha=0.5, label='± 1 std')
# plt.plot(list(range(len(means))), means)
# plt.xlabel('food_reward')

In [8]:
import numpy as np
from scipy.stats import poisson
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt


data=np.array(performance_reward_rate(model, task))

# Define the Poisson probability mass function (PMF)
def poisson_pmf(x, mu):
    return poisson.pmf(x, mu)

x_values = np.arange(0, np.max(data) + 1)
hist_values, bin_edges = np.histogram(data, bins=np.arange(0, np.max(data) + 2))
mu_fit, _ = curve_fit(poisson_pmf, x_values, hist_values, p0=[5])

# Visualize the original data and the fitted Poisson distribution
plt.hist(data, bins=np.arange(0, np.max(data) + 1), density=True, alpha=0.6, label='Data')
x = np.arange(0, np.max(data) + 1)
plt.plot(x, poisson_pmf(x, mu_fit), 'r-', lw=2, label='Fitted Poisson distribution')
plt.xlabel('Value')
plt.ylabel('Probability')
plt.title('Fit data to Poisson distribution')
plt.legend()
plt.show()

print("Estimated Poisson distribution parameter (mean):", mu_fit[0])

KeyboardInterrupt: 

In [28]:

def par(fun, paramls):
    with Pool() as pool:
        results = pool.map(fun, paramls)
    return results

def eval_agent(food_reward):
    # init
    task=AF()
    # assign values
    task.food_reward=food_reward
    task.time_in_game_reward=time_in_game_reward
    task.att_coeff=att_coeff
    task.att_temp=att_temp
    task.penalty_cost=penalty_cost
    # load
    model = PPO.load(f'ycstore/{food_reward}', env=task, device='cpu')
    model_list[food_reward]=model
    # eval
    return (performance_reward_rate(model, task,total_trials=100)[0])

with Pool() as pool:
    res = pool.map(eval_agent, food_reward_list)


# res=par(eval_agent, food_reward_list)
res

Wrapping the env with a `Monitor` wrapper
Wrapping the env with a `Monitor` wrapperWrapping the env with a `Monitor` wrapperWrapping the env in a DummyVecEnv.Wrapping the env with a `Monitor` wrapperWrapping the env with a `Monitor` wrapper



Wrapping the env in a DummyVecEnv.
Wrapping the env in a DummyVecEnv.Wrapping the env in a DummyVecEnv.
Wrapping the env in a DummyVecEnv.


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


KeyboardInterrupt: 

In [32]:
from stable_baselines3.common.vec_env import VecEnvWrapper, DummyVecEnv,SubprocVecEnv

env_list = [AF for _ in range(4)]
vec_env = SubprocVecEnv(env_list)
obs = vec_env.reset()
inference_data = []
for _ in range(1000): 
    while True:
        action, _ = model.predict(obs)
        obs, reward, done, _ = vec_env.step(action)
        # Collect inference data (e.g., observations, actions, rewards)
        inference_data.append((obs, action, reward, done))
        if done.any():
            obs = vec_env.reset()
            break
vec_env.close()

/home/yc/miniconda3/envs/ffneural/lib/python3.12/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/home/yc/miniconda3/envs/ffneural/lib/python3.12/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/home/yc/miniconda3/envs/ffneural/lib/python3.12/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automa